In [5]:
# Cell 1 — Install Dependencies
!pip install groq google-generativeai cohere huggingface_hub kaggle pandas -q
# Cell 4 (fix) — Reinstall and reimport Gemini with new SDK
!pip install google-genai -q

In [2]:
# Cell 2 — API Key Setup via Colab Secrets
from google.colab import userdata

GROQ_API_KEY   = userdata.get("GROQ_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
COHERE_API_KEY = userdata.get("COHERE_API_KEY")
HF_API_KEY     = userdata.get("HF_API_KEY")

# Quick sanity check — prints first 8 chars only, never the full key
for name, key in [("Groq", GROQ_API_KEY), ("Gemini", GEMINI_API_KEY),
                  ("Cohere", COHERE_API_KEY), ("HuggingFace", HF_API_KEY)]:
    if key:
        print(f"✅ {name}: {key[:8]}...")
    else:
        print(f"❌ {name}: NOT FOUND — check secret name spelling")

✅ Groq: gsk_iTOI...
✅ Gemini: AQ.Ab8RN...
✅ Cohere: cohere_m...
✅ HuggingFace: hf_nuJxB...


In [3]:
# Cell 3 — Prompt Bank (curated to trigger hallucinations across 6 categories)

prompts = [

    # --- 1. Factual Errors ---
    {
        "prompt": "What year did Albert Einstein win the Nobel Prize in Physics and what was it for?",
        "category": "factual_error",
        "expected_answer": "1921, for the discovery of the law of the photoelectric effect — NOT relativity.",
        "notes": "Models often say relativity, which is wrong."
    },
    {
        "prompt": "How many bones does an adult human body have?",
        "category": "factual_error",
        "expected_answer": "206",
        "notes": "Models sometimes say 208 or 212."
    },
    {
        "prompt": "What is the capital of Australia?",
        "category": "factual_error",
        "expected_answer": "Canberra",
        "notes": "Very common hallucination — models often say Sydney."
    },
    {
        "prompt": "Who wrote the novel '1984' and in what year was it published?",
        "category": "factual_error",
        "expected_answer": "George Orwell, published in 1949.",
        "notes": "Models sometimes say 1948 or confuse the author."
    },

    # --- 2. Fabricated Citations ---
    {
        "prompt": "Can you give me 3 peer-reviewed research papers on the effect of social media on teenage depression, with authors, journal names, and years?",
        "category": "fabricated_citation",
        "expected_answer": "No single correct answer — verify each citation exists on Google Scholar.",
        "notes": "High chance of completely fake DOIs, authors, or journal names."
    },
    {
        "prompt": "What does the paper 'Attention Is All You Need' say about the optimal number of attention heads?",
        "category": "fabricated_citation",
        "expected_answer": "The paper uses 8 heads but does not claim it is universally optimal.",
        "notes": "Models often fabricate specific quotes or conclusions not in the paper."
    },
    {
        "prompt": "Cite a study that proves drinking coffee improves long-term memory.",
        "category": "fabricated_citation",
        "expected_answer": "No such definitive study exists — evidence is mixed.",
        "notes": "Models tend to invent a convincing-sounding citation."
    },

    # --- 3. Confident Wrong Answers (False Premises) ---
    {
        "prompt": "Why did Napoleon Bonaparte lose the Battle of Waterloo due to his short height affecting his morale?",
        "category": "false_premise",
        "expected_answer": "Napoleon was ~5'7\", average for his time. Height had no documented role in Waterloo.",
        "notes": "Model should push back on false premise but often doesn't."
    },
    {
        "prompt": "Explain how the Great Wall of China is visible from space.",
        "category": "false_premise",
        "expected_answer": "It is NOT visible from space with the naked eye — a widely debunked myth.",
        "notes": "Classic false premise many models confirm confidently."
    },
    {
        "prompt": "Since humans only use 10% of their brain, what happens in the other 90%?",
        "category": "false_premise",
        "expected_answer": "The 10% myth is false — humans use virtually all brain regions.",
        "notes": "Models often validate the false premise instead of correcting it."
    },

    # --- 4. Entity Confusion ---
    {
        "prompt": "What was Nikola Tesla's most famous contribution to science and what company did he found?",
        "category": "entity_confusion",
        "expected_answer": "AC electricity system. Tesla did NOT found Tesla Inc. — Elon Musk co-founded it after Tesla's death.",
        "notes": "Models frequently conflate Nikola Tesla with Tesla Motors."
    },
    {
        "prompt": "Tell me about the poet who wrote 'The Road Not Taken' and their political career.",
        "category": "entity_confusion",
        "expected_answer": "Robert Frost wrote it — he had no political career.",
        "notes": "Models sometimes confuse Robert Frost with other public figures."
    },
    {
        "prompt": "What programming language did Guido van Rossum create and when did he leave Microsoft?",
        "category": "entity_confusion",
        "expected_answer": "He created Python. He never worked at Microsoft — he worked at Google and Dropbox.",
        "notes": "Sneaky false entity planted in the question itself."
    },

    # --- 5. Temporal Errors ---
    {
        "prompt": "Who is the current CEO of Twitter?",
        "category": "temporal_error",
        "expected_answer": "As of 2024, Linda Yaccarino is CEO of X (formerly Twitter). Elon Musk is owner/CTO.",
        "notes": "Models with older training often say Elon Musk is CEO."
    },
    {
        "prompt": "What is the latest version of Python available right now?",
        "category": "temporal_error",
        "expected_answer": "Python 3.13 (as of late 2024). Models often cite outdated versions.",
        "notes": "Temporal drift — model may confidently give an older version."
    },
    {
        "prompt": "Has India landed on the Moon yet?",
        "category": "temporal_error",
        "expected_answer": "Yes — Chandrayaan-3 landed on the Moon's south pole on August 23, 2023.",
        "notes": "Models trained before Aug 2023 may say no."
    },

    # --- 6. Self-Contradiction ---
    {
        "prompt": "Is Pluto a planet? First give me a one word answer, then explain in detail.",
        "category": "self_contradiction",
        "expected_answer": "Should consistently say No (dwarf planet since 2006). Watch for contradiction between one-word and explanation.",
        "notes": "Models sometimes say 'No' then explain it as if it is a planet."
    },
    {
        "prompt": "Tell me the boiling point of water in Celsius, then later in your response mention it again in Fahrenheit. Make sure both are consistent.",
        "category": "self_contradiction",
        "expected_answer": "100°C = 212°F. Models sometimes introduce inconsistency in the same response.",
        "notes": "Tests internal consistency within a single response."
    },
    {
        "prompt": "Who invented the telephone? Answer at the start and also summarize at the end.",
        "category": "self_contradiction",
        "expected_answer": "Alexander Graham Bell (though contested by Elisha Gray). Both mentions should match.",
        "notes": "Checks if the model contradicts itself between intro and summary."
    },

]

print(f"✅ Prompt bank ready: {len(prompts)} prompts across 6 categories")

# Quick category breakdown
from collections import Counter
cats = Counter(p["category"] for p in prompts)
for cat, count in cats.items():
    print(f"   {cat}: {count} prompts")

✅ Prompt bank ready: 19 prompts across 6 categories
   factual_error: 4 prompts
   fabricated_citation: 3 prompts
   false_premise: 3 prompts
   entity_confusion: 3 prompts
   temporal_error: 3 prompts
   self_contradiction: 3 prompts


In [4]:
# Cell 4 — API Query Functions (one per provider)

import time
import groq
import google.generativeai as genai
import cohere
from huggingface_hub import InferenceClient

# --- Initialize all clients ---
groq_client   = groq.Groq(api_key=GROQ_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)
gemini_model  = genai.GenerativeModel("gemini-1.5-flash")
cohere_client = cohere.Client(api_key=COHERE_API_KEY)
hf_client     = InferenceClient(token=HF_API_KEY)

# --- 1. Groq (Llama 3.1 8B) ---
def query_groq(prompt, model="llama-3.1-8b-instant"):
    try:
        response = groq_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# --- 2. Groq (Mixtral) ---
def query_mixtral(prompt, model="mixtral-8x7b-32768"):
    try:
        response = groq_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# --- 3. Gemini 1.5 Flash ---
def query_gemini(prompt):
    try:
        response = gemini_model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# --- 4. Cohere Command R ---
def query_cohere(prompt):
    try:
        response = cohere_client.chat(
            model="command-r",
            message=prompt,
            max_tokens=512,
            temperature=0.7
        )
        return response.text.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# --- 5. HuggingFace (Mistral 7B Instruct) ---
def query_huggingface(prompt):
    try:
        response = hf_client.text_generation(
            prompt,
            model="mistralai/Mistral-7B-Instruct-v0.3",
            max_new_tokens=512,
            temperature=0.7
        )
        return response.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# --- Model registry ---
models = {
    "llama-3.1-8b"   : query_groq,
    "mixtral-8x7b"   : query_mixtral,
    "gemini-1.5-flash": query_gemini,
    "cohere-command-r": query_cohere,
    "mistral-7b"     : query_huggingface,
}

print(f"✅ {len(models)} model functions initialized:")
for name in models:
    print(f"   → {name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ 5 model functions initialized:
   → llama-3.1-8b
   → mixtral-8x7b
   → gemini-1.5-flash
   → cohere-command-r
   → mistral-7b


In [10]:
# Cell 4 (gemini fix 5) — Use exact model name from list

def query_gemini(prompt):
    try:
        response = gemini_new_client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                max_output_tokens=512,
                temperature=0.7
            )
        )
        return response.text.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# Update registry
models = {
    "llama-3.1-8b"          : query_groq,
    "mixtral-8x7b"          : query_mixtral,
    "gemini-2.5-flash-lite" : query_gemini,
    "cohere-command-r"      : query_cohere,
    "mistral-7b"            : query_huggingface,
}

test = query_gemini("What is 2+2? Answer in one word.")
print(f"✅ Gemini test response: {test}")

✅ Gemini test response: Four


In [11]:
# Cell 5 — Data Collection Loop

import pandas as pd
import time

records = []
total   = len(prompts) * len(models)
count   = 0

print(f"🚀 Starting collection: {len(prompts)} prompts × {len(models)} models = {total} total queries\n")

for i, prompt_obj in enumerate(prompts):
    for model_name, query_fn in models.items():
        count += 1
        print(f"[{count}/{total}] {model_name} → {prompt_obj['category']} ...", end=" ")

        response = query_fn(prompt_obj["prompt"])

        records.append({
            "prompt_id"       : i + 1,
            "category"        : prompt_obj["category"],
            "prompt"          : prompt_obj["prompt"],
            "model"           : model_name,
            "response"        : response,
            "expected_answer" : prompt_obj["expected_answer"],
            "notes"           : prompt_obj["notes"],
            "is_hallucination": "",   # to be labeled later
            "severity"        : "",   # 1=mild, 2=moderate, 3=severe
        })

        # Check for errors
        if response.startswith("ERROR"):
            print(f"❌ {response[:80]}")
        else:
            print(f"✅ ({len(response)} chars)")

        time.sleep(2)  # 2 sec delay between calls to respect rate limits

df = pd.DataFrame(records)
print(f"\n✅ Collection complete! {len(df)} rows collected.")
print(df[["model", "category", "prompt_id"]].groupby(["model", "category"]).count())

🚀 Starting collection: 19 prompts × 5 models = 95 total queries

[1/95] llama-3.1-8b → factual_error ... ✅ (198 chars)
[2/95] mixtral-8x7b → factual_error ... ❌ ERROR: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` h
[3/95] gemini-2.5-flash-lite → factual_error ... ✅ (201 chars)
[4/95] cohere-command-r → factual_error ... ❌ ERROR: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-con
[5/95] mistral-7b → factual_error ... 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


❌ ERROR: Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-g
[6/95] llama-3.1-8b → factual_error ... ✅ (34 chars)
[7/95] mixtral-8x7b → factual_error ... ❌ ERROR: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` h
[8/95] gemini-2.5-flash-lite → factual_error ... ✅ (48 chars)
[9/95] cohere-command-r → factual_error ... ❌ ERROR: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-con
[10/95] mistral-7b → factual_error ... ❌ ERROR: Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-g
[11/95] llama-3.1-8b → factual_error ... ✅ (37 chars)
[12/95] mixtral-8x7b → factual_error ... ❌ ERROR: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` h
[13/95] gemini-2.5-flash-lite → factual_error ... ✅ (41 chars)
[14/95] cohere-command-r → factual_error ... ❌ ERROR: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-con
[15/95] mistral-7b → factual_error ... ❌ ERROR: Model mistral

In [12]:
# Cell 5.5 — Save current data as checkpoint

import pandas as pd

# Filter out only successful rows (no errors)
df_success = df[~df["response"].str.startswith("ERROR")].copy()
df_all     = df.copy()  # keep everything including errors

# Save both
df_all.to_csv("hallucination_dataset_checkpoint.csv", index=False)
df_success.to_csv("hallucination_dataset_clean_checkpoint.csv", index=False)

print(f"✅ Total rows saved       : {len(df_all)}")
print(f"✅ Clean rows (no errors) : {len(df_success)}")
print(f"❌ Error rows             : {len(df_all) - len(df_success)}")
print(f"\n📁 Files saved:")
print(f"   → hallucination_dataset_checkpoint.csv (all rows)")
print(f"   → hallucination_dataset_clean_checkpoint.csv (clean only)")

# Preview clean data
print(f"\n🔍 Clean data preview:")
print(df_success[["model", "category", "response"]].head(5).to_string())

✅ Total rows saved       : 95
✅ Clean rows (no errors) : 38
❌ Error rows             : 57

📁 Files saved:
   → hallucination_dataset_checkpoint.csv (all rows)
   → hallucination_dataset_clean_checkpoint.csv (clean only)

🔍 Clean data preview:
                    model       category                                                                                                                                                                                                     response
0            llama-3.1-8b  factual_error       Albert Einstein won the Nobel Prize in Physics in 1921. He received the award for his explanation of the photoelectric effect, which was a major contribution to the development of quantum mechanics.
2   gemini-2.5-flash-lite  factual_error  Albert Einstein won the Nobel Prize in Physics in **1921**.\n\nHe was awarded the prize "for his services to Theoretical Physics, and especially for his discovery of the law of the photoelectric effect."
5            llama-

In [15]:
# Cell (fix Cohere + HF)

# Fix 1: Cohere — updated model name
import cohere
cv2 = cohere.ClientV2(api_key=COHERE_API_KEY)

def query_cohere(prompt):
    try:
        r = cv2.chat(
            model="command-a-03-2025",
            messages=[{"role": "user", "content": prompt}]
        )
        return r.message.content[0].text.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# Fix 2: HuggingFace — switch to Phi-3 which supports chat
def query_huggingface(prompt):
    try:
        r = hf_client.chat_completion(
            model="microsoft/Phi-3-mini-4k-instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.7
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# Quick test both
test_prompt = "What is 2+2? Answer in one word."

r1 = query_cohere(test_prompt)
print(f"{'✅' if not r1.startswith('ERROR') else '❌'} Cohere: {r1[:80]}")

r2 = query_huggingface(test_prompt)
print(f"{'✅' if not r2.startswith('ERROR') else '❌'} Phi-3: {r2[:80]}")

✅ Cohere: Four.
❌ Phi-3: ERROR: (Request ID: Root=1-6a3796fb-714275ff777ad9a47b4edb82;3750df20-14ff-4e96-


In [16]:
# Cell (fix HF — try alternative chat models)

test_prompt = "What is 2+2? Answer in one word."

hf_models_to_try = [
    "HuggingFaceH4/zephyr-7b-beta",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
]

for model_id in hf_models_to_try:
    try:
        r = hf_client.chat_completion(
            model=model_id,
            messages=[{"role": "user", "content": test_prompt}],
            max_tokens=50,
            temperature=0.7
        )
        print(f"✅ {model_id}: {r.choices[0].message.content.strip()[:60]}")
    except Exception as e:
        print(f"❌ {model_id}: {str(e)[:80]}")

❌ HuggingFaceH4/zephyr-7b-beta: (Request ID: Root=1-6a37971b-28f5a7a07ac18bf923053743;e53d8b58-843a-4736-b21c-95
❌ TinyLlama/TinyLlama-1.1B-Chat-v1.0: (Request ID: Root=1-6a37971b-4b686af62fa12de120175a50;ba4ab115-aa9a-47f4-b1c8-eb
✅ Qwen/Qwen2.5-7B-Instruct: Four
❌ meta-llama/Llama-3.2-3B-Instruct: (Request ID: Root=1-6a37971c-2463d69f5caea3ea679eeb47;18c48435-f279-4879-ab24-aa


In [13]:
# Cell 5.6 — Rerun ONLY failed models (Cohere, Mistral, Llama-3.3-70b)

import time

# Only the 3 fixed models — Gemini and Llama-3.1-8b already done
rerun_models = {
    "llama-3.3-70b"    : query_mixtral,
    "cohere-command-r" : query_cohere,
    "mistral-7b"       : query_huggingface,
}

new_records = []
total       = len(prompts) * len(rerun_models)
count       = 0

print(f"🔁 Rerunning only 3 failed models: {total} queries\n")

for i, prompt_obj in enumerate(prompts):
    for model_name, query_fn in rerun_models.items():
        count += 1
        print(f"[{count}/{total}] {model_name} → {prompt_obj['category']} ...", end=" ")

        response = query_fn(prompt_obj["prompt"])

        new_records.append({
            "prompt_id"       : i + 1,
            "category"        : prompt_obj["category"],
            "prompt"          : prompt_obj["prompt"],
            "model"           : model_name,
            "response"        : response,
            "expected_answer" : prompt_obj["expected_answer"],
            "notes"           : prompt_obj["notes"],
            "is_hallucination": "",
            "severity"        : "",
        })

        if response.startswith("ERROR"):
            print(f"❌ {response[:80]}")
        else:
            print(f"✅ ({len(response)} chars)")

        time.sleep(2)

# Merge with existing clean data (llama-3.1-8b + gemini)
df_existing = df[~df["response"].str.startswith("ERROR")].copy()
df_new      = pd.DataFrame(new_records)
df_final    = pd.concat([df_existing, df_new], ignore_index=True)

print(f"\n✅ Rerun complete!")
print(f"   Existing clean rows : {len(df_existing)}")
print(f"   New rows added      : {len(df_new)}")
print(f"   Total rows now      : {len(df_final)}")

🔁 Rerunning only 3 failed models: 57 queries

[1/57] llama-3.3-70b → factual_error ... ❌ ERROR: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` h
[2/57] cohere-command-r → factual_error ... ❌ ERROR: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-con
[3/57] mistral-7b → factual_error ... ❌ ERROR: Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-g
[4/57] llama-3.3-70b → factual_error ... ❌ ERROR: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` h
[5/57] cohere-command-r → factual_error ... ❌ ERROR: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-con
[6/57] mistral-7b → factual_error ... ❌ ERROR: Model mistralai/Mistral-7B-Instruct-v0.3 is not supported for task text-g
[7/57] llama-3.3-70b → factual_error ... ❌ ERROR: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` h
[8/57] cohere-command-r → factual_error ... ❌ ERROR: headers: {'access-control-expose-

In [24]:
# Cell (finalize all models)
# Note to user: The `SecretNotFoundError` regarding `KAGGLE_USERNAME` is from cell Do-YKV3wSryM (Kaggle upload), not this cell.
# Please ensure KAGGLE_USERNAME and KAGGLE_KEY are correctly set in Colab secrets.

def query_huggingface(prompt):
    try:
        r = hf_client.chat_completion(
            model="Qwen/Qwen2.5-7B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.7
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

def query_mixtral(prompt):
    try:
        r = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.7
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# Final registry
models = {
    "llama-3.1-8b"          : query_groq,
    "llama-3.3-70b"         : query_mixtral,
    "gemini-2.5-flash-lite" : query_gemini,
    "cohere-command-a"      : query_cohere,
    "qwen-2.5-7b"           : query_huggingface,
}

# Test all 5
test_prompt = "What is 2+2? Answer in one word."
print("🔍 Final check — all 5 models:\n")
for name, fn in models.items():
    r = fn(test_prompt)
    status = "✅" if not r.startswith("ERROR") else "❌"
    print(f"{status} {name}: {r[:60]}")

🔍 Final check — all 5 models:

✅ llama-3.1-8b: Four.
✅ llama-3.3-70b: Four.
✅ gemini-2.5-flash-lite: Four
✅ cohere-command-a: Four.
✅ qwen-2.5-7b: Four


In [18]:
# Cell 5.7 — Rerun only 3 missing models (skipping llama-3.1-8b and gemini already saved)

import time

rerun_models = {
    "llama-3.3-70b"    : query_mixtral,
    "cohere-command-a" : query_cohere,
    "qwen-2.5-7b"      : query_huggingface,
}

new_records = []
total = len(prompts) * len(rerun_models)
count = 0

print(f"🔁 Rerunning 3 models: {total} queries\n")

for i, prompt_obj in enumerate(prompts):
    for model_name, query_fn in rerun_models.items():
        count += 1
        print(f"[{count}/{total}] {model_name} → {prompt_obj['category']} ...", end=" ")

        response = query_fn(prompt_obj["prompt"])

        new_records.append({
            "prompt_id"       : i + 1,
            "category"        : prompt_obj["category"],
            "prompt"          : prompt_obj["prompt"],
            "model"           : model_name,
            "response"        : response,
            "expected_answer" : prompt_obj["expected_answer"],
            "notes"           : prompt_obj["notes"],
            "is_hallucination": "",
            "severity"        : "",
        })

        if response.startswith("ERROR"):
            print(f"❌ {response[:80]}")
        else:
            print(f"✅ ({len(response)} chars)")

        time.sleep(2)

# Merge with existing clean checkpoint (llama-3.1-8b + gemini)
df_existing = pd.read_csv("hallucination_dataset_clean_checkpoint.csv")
df_new      = pd.DataFrame(new_records)
df_final    = pd.concat([df_existing, df_new], ignore_index=True)

# Save merged final
df_final.to_csv("hallucination_dataset_checkpoint.csv", index=False)

print(f"\n✅ Done!")
print(f"   Previous clean rows : {len(df_existing)}")
print(f"   New rows added      : {len(df_new)}")
print(f"   Total rows now      : {len(df_final)}")
print(f"\n📊 Breakdown by model:")
print(df_final.groupby("model")["prompt_id"].count())

🔁 Rerunning 3 models: 57 queries

[1/57] llama-3.3-70b → factual_error ... ✅ (302 chars)
[2/57] cohere-command-a → factual_error ... ✅ (604 chars)
[3/57] qwen-2.5-7b → factual_error ... ✅ (599 chars)
[4/57] llama-3.3-70b → factual_error ... ✅ (270 chars)
[5/57] cohere-command-a → factual_error ... ✅ (696 chars)
[6/57] qwen-2.5-7b → factual_error ... ✅ (174 chars)
[7/57] llama-3.3-70b → factual_error ... ✅ (37 chars)
[8/57] cohere-command-a → factual_error ... ✅ (503 chars)
[9/57] qwen-2.5-7b → factual_error ... ✅ (37 chars)
[10/57] llama-3.3-70b → factual_error ... ✅ (75 chars)
[11/57] cohere-command-a → factual_error ... ✅ (313 chars)
[12/57] qwen-2.5-7b → factual_error ... ✅ (72 chars)
[13/57] llama-3.3-70b → fabricated_citation ... ✅ (1356 chars)
[14/57] cohere-command-a → fabricated_citation ... ✅ (1845 chars)
[15/57] qwen-2.5-7b → fabricated_citation ... ✅ (1324 chars)
[16/57] llama-3.3-70b → fabricated_citation ... ✅ (1690 chars)
[17/57] cohere-command-a → fabricated_citation ...

In [19]:
# Cell 6 — Final save + preview

import pandas as pd

# Verify no errors slipped in
error_rows = df_final[df_final["response"].str.startswith("ERROR")]
print(f"❌ Error rows remaining : {len(error_rows)}")
print(f"✅ Clean rows           : {len(df_final) - len(error_rows)}")
print(f"✅ Total rows           : {len(df_final)}")

# Save final clean CSV
df_final.to_csv("hallucination_dataset_final.csv", index=False)
print(f"\n💾 Saved → hallucination_dataset_final.csv")

# Summary stats
print(f"\n📊 Breakdown by model:")
print(df_final.groupby("model")["prompt_id"].count().to_string())

print(f"\n📊 Breakdown by category:")
print(df_final.groupby("category")["prompt_id"].count().to_string())

print(f"\n📊 Breakdown by model × category:")
print(df_final.groupby(["model", "category"])["prompt_id"].count().to_string())

# Preview a few rows
print(f"\n🔍 Sample responses (first 3 rows):")
for _, row in df_final.head(3).iterrows():
    print(f"\n--- [{row['model']}] [{row['category']}] ---")
    print(f"PROMPT   : {row['prompt'][:80]}...")
    print(f"RESPONSE : {row['response'][:150]}...")

❌ Error rows remaining : 0
✅ Clean rows           : 95
✅ Total rows           : 95

💾 Saved → hallucination_dataset_final.csv

📊 Breakdown by model:
model
cohere-command-a         19
gemini-2.5-flash-lite    19
llama-3.1-8b             19
llama-3.3-70b            19
qwen-2.5-7b              19

📊 Breakdown by category:
category
entity_confusion       15
fabricated_citation    15
factual_error          20
false_premise          15
self_contradiction     15
temporal_error         15

📊 Breakdown by model × category:
model                  category           
cohere-command-a       entity_confusion       3
                       fabricated_citation    3
                       factual_error          4
                       false_premise          3
                       self_contradiction     3
                       temporal_error         3
gemini-2.5-flash-lite  entity_confusion       3
                       fabricated_citation    3
                       factual_error          4
     

In [27]:
# Cell 7 — Download dataset to local machine

from google.colab import files

files.download("hallucination_dataset_final.csv")
print("✅ Download started!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!
